In [ ]:
from google import genai
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

sentence = "I love playing football."

# ask to convert the sentence into a vector
result = client.models.embed_content(
    model="gemini-embedding-001",  # the specific AI model that does this conversion
    contents=sentence
)

# Pulls the actual list of numbers out of the result
embedding = result.embeddings[0].values

print("Sentence:", sentence)
print("First 5 numbers of its vector:", embedding[:5])  # just a peek, the full list is huge
print("Total numbers in the vector:", len(embedding))   # usually 768 or 3072 numbers

Sentence: I love playing football.
First 5 numbers of its vector: [-0.012424914, 0.0066163046, 0.021517763, -0.06809436, -0.01598877]
Total numbers in the vector: 3072


In [10]:


pip install google-genai python-dotenv scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from google import genai
from dotenv import load_dotenv
import os
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

# A tiny helper function so we don't repeat the same 4 lines every time
def get_embedding(text):
    # Send text to the model, get back numbers
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text
    )
    # Return just the list of numbers
    return result.embeddings[0].values

sentence1 = "I love playing football."
sentence2 = "I enjoy playing soccer."

# Convert both to numbers using our helper function
vector1 = get_embedding(sentence1)
vector2 = get_embedding(sentence2)

similarity = cosine_similarity([vector1], [vector2])

score = similarity[0][0]

print("Sentence 1:", sentence1)
print("Sentence 2:", sentence2)
print("How similar are they? (closer to 1 = more similar):", score)

Sentence 1: I love playing football.
Sentence 2: I enjoy playing soccer.
How similar are they? (closer to 1 = more similar): 0.8180875413935074


In [ ]:
from google import genai
from dotenv import load_dotenv
import os
import time
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

def get_embedding(text):
    result = client.models.embed_content(model="gemini-embedding-001", contents=text)
    return result.embeddings[0].values

sentences = [
    "I love playing football.",
    "Python is a programming language.",
    "I enjoy playing soccer."
]

# one vector per sentence, so we loop through and collect them
embeddings = []
for sentence in sentences:
    vector = get_embedding(sentence)
    embeddings.append(vector)
    time.sleep(2)

# compare EVERY sentence to EVERY other sentence in one go
similarity_matrix = cosine_similarity(embeddings)

print("Sentences:")
for i, sentence in enumerate(sentences):
    print(i, "-", sentence)

print("\nSimilarity grid:")
print(similarity_matrix)

Sentences:
0 - I love playing football.
1 - Python is a programming language.
2 - I enjoy playing soccer.

Similarity grid:
[[1.         0.57107916 0.81808754]
 [0.57107916 1.         0.60288585]
 [0.81808754 0.60288585 1.        ]]


In [ ]:
from google import genai
from dotenv import load_dotenv
import os
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

def get_embedding(text):
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text
        )
    return result.embeddings[0].values

sentences = [
    "I love playing football.",
    "Python is a programming language.",
    "I enjoy playing soccer."
]

sentence_embeddings = [get_embedding(s) for s in sentences]

def search(query):
    query_vector = get_embedding(query)                                     # Step A: turn the user's question into numbers too

    scores = cosine_similarity([query_vector], sentence_embeddings)[0]      # Step B: compare the question's numbers against EVERY sentence's numbers

    paired = list(zip(sentences, scores))                                   # Step C: glue each sentence to its score, so we don't lose track of which score belongs to which sentence

    paired.sort(key=lambda pair: pair[1], reverse=True)                     # Step D: sort so the HIGHEST score (best match) comes first

    print(f"\nYou searched for: '{query}'")                                 # Step E: print everything, best match first
    for sentence, score in paired:
        print(f"  score={score:.3f}  ->  {sentence}")

    best_sentence, best_score = paired[0]                                   # The best match is just the first item after sorting
    print(f"\nBest match: {best_sentence}")

search("I like sports")
search("Tell me about coding")


You searched for: 'I like sports'
  score=0.750  ->  I enjoy playing soccer.
  score=0.738  ->  I love playing football.
  score=0.581  ->  Python is a programming language.

Best match: I enjoy playing soccer.

You searched for: 'Tell me about coding'
  score=0.592  ->  Python is a programming language.
  score=0.567  ->  I love playing football.
  score=0.561  ->  I enjoy playing soccer.

Best match: Python is a programming language.
